# Document parsing

In [1]:
from docx import Document

doc = Document("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx")
doc

In [2]:
def extract_only_bullets(docx_path):
    doc = Document(docx_path)
    bullet_points = []
    
    for paragraph in doc.paragraphs:
        pPr = paragraph._element.pPr
        
        if pPr is not None:
            numPr = pPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numPr')
            
            if numPr is not None:
                # Get the numId
                numId = numPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numId')
                
                if numId is not None:
                    num_id = numId.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')
                    
                    # Check if it's a bullet (not numbered)
                    if is_bullet_list(doc, num_id):
                        bullet_points.append(paragraph.text)
    
    return bullet_points

def is_bullet_list(doc, num_id):
    """Check if a numbering ID corresponds to a bullet list"""
    try:
        numbering_part = doc.part.numbering_part
        numbering_xml = numbering_part.element
        
        # Find the num element with this numId
        num = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}num[@{{{numbering_xml.nsmap["w"]}}}numId="{num_id}"]')
        
        if num is not None:
            # Get the abstractNumId
            abstractNumId = num.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNumId')
            abstract_num_id = abstractNumId.get(f'{{{numbering_xml.nsmap["w"]}}}val')
            
            # Find the abstractNum definition
            abstractNum = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNum[@{{{numbering_xml.nsmap["w"]}}}abstractNumId="{abstract_num_id}"]')
            
            if abstractNum is not None:
                # Check the first level's numFmt
                lvl = abstractNum.find(f'.//{{{numbering_xml.nsmap["w"]}}}lvl[@{{{numbering_xml.nsmap["w"]}}}ilvl="0"]')
                numFmt = lvl.find(f'.//{{{numbering_xml.nsmap["w"]}}}numFmt')
                
                # Bullet lists use 'bullet' format, numbered use 'decimal', 'lowerLetter', etc.
                fmt_val = numFmt.get(f'{{{numbering_xml.nsmap["w"]}}}val')
                return fmt_val == 'bullet'
    except:
        pass
    
    return False

In [3]:
bullets = extract_only_bullets("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx")
bullets

['1950: The Haemophilia Society (UK) was formed.',
 'Late 1980s: Early Treatments: Interferon was the first antiviral treatment for hepatitis C.  significant side effects and lower cure rates.',
 'Late 1980s: Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections',
 '1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff.  Early 1990s expanded to a national level. ',
 'November 1987: announcement of £10 million payment for haemophiliacs infected with HIV',
 '7 March 1988: British Liver Trust incorporated and established.  (Founded by hepatologist Professor Dame Sheila Sherlock, leading charity for adults, focusing on raising awareness, p

In [8]:
import re
from collections import defaultdict

months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)'

date_pattern = r'\b(?:' \
    rf'\d{{1,2}}\s+{months}\s+\d{{4}}|' \
    rf'{months}\s+\d{{1,2}},?\s+\d{{4}}|' \
    r'(?:Early|Mid|Late)\s+\d{4}s?|' \
    rf'{months}\s+\d{{4}}' \
    r')\b'

def extract_date(text):
    """Extract the date string from text"""
    match = re.search(date_pattern, text, re.IGNORECASE)
    if match:
        return match.group(0)
    return None

def remove_date(text, date_str):
    """Remove the date from the text and clean up extra whitespace"""
    cleaned = text.replace(date_str, '').strip()
    # Clean up any double spaces or extra punctuation left behind
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'^\s*[-:]\s*', '', cleaned)  # Remove leading dash or colon
    return cleaned

def group_bullets_by_date_dict(bullet_list):
    """Create dictionary with dates as keys and associated text as values"""
    date_dict = defaultdict(list)
    current_date = None
    
    for bullet in bullet_list:
        date_str = extract_date(bullet)
        
        if date_str:
            # This bullet has a date - start a new entry
            current_date = date_str
            # Remove date from text and make it the first element
            text_without_date = remove_date(bullet, date_str)
            date_dict[current_date].append(text_without_date)
        else:
            # This bullet doesn't have a date - add to current date's list
            if current_date:
                date_dict[current_date].append(bullet)
    
    return dict(date_dict)


In [9]:
grouped = group_bullets_by_date_dict(bullets)
grouped, len(grouped)

({'Late 1980s': ['Early Treatments: Interferon was the first antiviral treatment for hepatitis C. significant side effects and lower cure rates.',
   'Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections'],
  'Early 1990s': ['1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff. expanded to a national level.'],
  'November 1987': ['announcement of £10 million payment for haemophiliacs infected with HIV'],
  '7 March 1988': ['British Liver Trust incorporated and established. (Founded by hepatologist Professor Dame Sheila Sherlock, leading charity for adults, focusing on raising awareness, providing support, and campaigning for improve

# Retrieval

In [1]:
import pandas as pd

df = pd.read_csv("./all_ratings811.csv")
def parse_date(val):
    for fmt in ('%Y-%m-%d', '%B %d, %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError) as e:
            continue
    return pd.NaT  # fallback if nothing matches

df['Date'] = df['Date'].apply(lambda x: x.replace(" 00:00:00", "")).apply(parse_date)
df = df[["Title", "Date", "Author", "Newspaper", "Source", "newspaper_type", "chunks"]]

In [2]:
df

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
0,International Companies: Technology - Pounds 2...,1983-02-11,DAVID FISHLOCK,Financial Times,Factiva - HepC,broadsheet,BRITAIN'S biotechnologists are to have a new P...
1,Study confirms fears on spread of Aids,1984-08-31,ANDREW VEITCH,The Guardian,Lexis-Nexis - Times,broadsheet,Fears that the Aids virus is widespread among ...
2,Maternity unit where waste parades the corrido...,1984-10-20,No author in byline,The Guardian,Lexis-Nexis - Times,broadsheet,Dawn in the Rosie maternity unit at the 900-be...
3,Prisons inspector condemns sanitary conditions...,1984-10-24,AILEEN BALLANTYNE,The Guardian,Lexis-Nexis - Times,broadsheet,Gaol conditions fall short of the standard exp...
4,Financial Times Survey: China - Business - Gro...,1984-10-29,No author in byline,Financial Times,Factiva - HepC,broadsheet,CLIMATENorth China (including Beijing) - Winte...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("/Users/pnadel01/fp_models/google_embeddinggemma-300m", device="mps")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [8]:
# test key: '29 August 2003'

sent_sim_template = "task: sentence similarity | query: {content}"

# date_key_from_timetable = '29 August 2003'
# associated_text = grouped[date_key_from_timetable]
# query_embedding = model.encode(sent_sim_template.format(content=associated_text))
# query_embedding.shape

In [15]:
times = df[df.Newspaper == "The Times"]
times

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
7,Travel: How to stay healthy on your summer bre...,1985-07-06,ISABEL WALKER,The Times,Factiva - HepC,broadsheet,Holidays are one of the 30 most stressful even...
8,Family Money: Taking the sting from jabs / Ino...,1985-08-03,VIVIEN GOLDSMITH,The Times,Factiva - HepC,broadsheet,The series of jabs you will need if you are of...
9,A BREAKDOWN OF THE BREAKDOWNS (732),1985-08-14,JOHN GOODBODY,The Times,Factiva - HepC,broadsheet,Steve Cram1981Has a winter of injuries.August:...
10,Times Diary: Les-majeste (105),1985-08-30,PHS,The Times,Factiva - HepC,broadsheet,After years of pretending to find Barry Humphr...
13,Books: Travelling pros(e) on the exotic job / ...,1985-11-28,ALLAN MASSIE,The Times,Factiva - HepC,broadsheet,Muddling Through In Madagascar by Dervla Murph...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [16]:
key_dt = pd.to_datetime(date_key_from_timetable)
buffer = pd.DateOffset(years=1)

zone_of_interest = times[
    (times.Date >= key_dt - buffer) & # backward looking
    (times.Date <= key_dt + buffer)   # forward looking
]
zone_of_interest 

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
1951,Lawsuit reassurances fail to revive Weston - M...,2002-08-31,No author in byline,The Times,Factiva - HepC,broadsheet,REASSURING comments that Weston Medical's need...
1954,Body piercing craze 'is putting teenagers' hea...,2002-09-10,Sam Lister and Patrick Barkham.,The Times,Factiva - HepC,broadsheet,"Body piercings below the neck, once the preser..."
1955,Gilt of the body politic - Cover story.,2002-09-11,Theodore Dalrymple.,The Times,Factiva - HepC,broadsheet,As tough new restrictions are proposed on body...
1956,The Times Vet - Pets.,2002-09-14,James Allcock.,The Times,Factiva - HepC,broadsheet,Q You answered a question about leg amputation...
1960,Liver failure death due to Ecstasy - In brief.,2002-09-17,No author in byline,The Times,Factiva - HepC,broadsheet,A 14-year-old girl suffered severe liver failu...
...,...,...,...,...,...,...,...
7433,", Saturday\n\n Shopping trips",2004-06-12,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 12, 2004, Saturday\..."
7434,", Friday\n\n What's Up, Doc?",2004-06-18,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 18, 2004, Friday\n\..."
7435,", Thursday\n\n So, anything to declare?",2004-07-29,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJuly 29, 2004, Thursday\..."
7436,", Tuesday\n\n Could meat give you Alzheimer's?",2004-08-10,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 10, 2004, Tuesday..."


In [17]:
doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
doc_embeddings.shape

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(121, 768)

In [18]:
sim_scores = doc_embeddings @ query_embedding.T
rel_ids_and_scores =[(i, sim_scores[i]) for i in (sim_scores).argsort()[::-1]]
rel_ids_and_scores

[(np.int64(65), np.float32(0.7566832)),
 (np.int64(50), np.float32(0.7512808)),
 (np.int64(112), np.float32(0.750149)),
 (np.int64(49), np.float32(0.7292455)),
 (np.int64(22), np.float32(0.71006584)),
 (np.int64(94), np.float32(0.68818104)),
 (np.int64(12), np.float32(0.67302406)),
 (np.int64(55), np.float32(0.6703769)),
 (np.int64(26), np.float32(0.66509414)),
 (np.int64(30), np.float32(0.657603)),
 (np.int64(98), np.float32(0.64630866)),
 (np.int64(92), np.float32(0.6392649)),
 (np.int64(66), np.float32(0.63884985)),
 (np.int64(20), np.float32(0.63880205)),
 (np.int64(27), np.float32(0.6387496)),
 (np.int64(16), np.float32(0.6191523)),
 (np.int64(97), np.float32(0.6155448)),
 (np.int64(75), np.float32(0.60481125)),
 (np.int64(38), np.float32(0.60325783)),
 (np.int64(59), np.float32(0.59671074)),
 (np.int64(107), np.float32(0.59583867)),
 (np.int64(68), np.float32(0.5955117)),
 (np.int64(33), np.float32(0.5744972)),
 (np.int64(41), np.float32(0.5735586)),
 (np.int64(21), np.float32(0.

In [19]:
import numpy as np

def scale_score(sim_scores, item_dates, target_date, scaler_func, date_weight):
    sim_scores = np.array(sim_scores)
    days_diffs = np.abs(np.array([(target_date - d).days for d in item_dates]))
    date_scores = np.array([scaler_func(d) for d in days_diffs])
    composite = (1 - date_weight) * sim_scores + date_weight * date_scores
    return composite, date_scores

def inverse_square(days_diff, max_days=365*2):
    normalized_diff = days_diff / max_days
    return 1 / (1 + normalized_diff ** 2)

In [20]:
composite, date_scores = scale_score(sim_scores, zone_of_interest.Date, key_dt, inverse_square, 0.5)
composite, date_scores 

(array([0.63313717, 0.60922786, 0.55187218, 0.62564215, 0.61986936,
        0.64671886, 0.61361636, 0.60399444, 0.56903819, 0.60345983,
        0.6917313 , 0.68657356, 0.76632209, 0.61110959, 0.66642766,
        0.59193776, 0.75633707, 0.72955954, 0.72195708, 0.62931908,
        0.77566978, 0.74199704, 0.81648133, 0.65916146, 0.6118918 ,
        0.66339324, 0.80328683, 0.79562996, 0.66136626, 0.71856554,
        0.80696516, 0.7417867 , 0.71299678, 0.77255995, 0.70150345,
        0.71523417, 0.6815965 , 0.74144122, 0.78972555, 0.71902697,
        0.69361535, 0.77896202, 0.67501675, 0.77019896, 0.70182789,
        0.75365121, 0.72516232, 0.72241352, 0.73512606, 0.8646218 ,
        0.87563945, 0.77069197, 0.71821433, 0.75197838, 0.71416696,
        0.83422951, 0.7531749 , 0.7104225 , 0.70514935, 0.79465894,
        0.70990356, 0.67314103, 0.73134059, 0.75861481, 0.72596642,
        0.85860128, 0.7986481 , 0.68546126, 0.77288998, 0.74223986,
        0.60553774, 0.70168495, 0.70822513, 0.73

In [21]:
from pprint import pprint

i = composite.argsort()[::-1][0]

print(i)
print(composite[i])
print(zone_of_interest.iloc[i].Title)
print(zone_of_interest.iloc[i].Date)
pprint(zone_of_interest.iloc[i].chunks)
print()
pprint(associated_text)

50
0.8756394540428893
Cash hopes for hepatitis victims
2003-08-30 00:00:00
('THE campaign to win compensation payments for Scots infected with hepatitis '
 'C from infected NHS blood products took an important step forward yesterday '
 'as the Scottish Executive announced that the UK Government had agreed that '
 'such payments could be made by the devolved body.Malcolm Chisholm, '
 "Scotland's Health Minister, said that a similar scheme was being established "
 'in England by the Department of Health.A stumbling block to implementing any '
 'such scheme remains, however, because the Department of Work and Pensions in '
 'Whitehall has given no guarantee that it will not claw back social security '
 'payments from hepatitis victims given compensation.Under the proposed scheme '
 'in Scotland, ex gratia payments of Pounds 20,000 would be made to all those '
 'who have the hepatitis C virus as a result of infected NHS blood products. A '
 'further Pounds 25,000 would go to to those in an

# Expand to dataset

In [27]:
from pprint import pprint
def time_bounded_search(df, model, date_key_from_timetable, month_threshold=2, go_backwards=True, scale_fn=inverse_square, date_weight=0.5, sim_threshold=.75, newspaper=None):
    if newspaper:
        df = df[df.Newspaper == newspaper]
        print(f"Filtered dataframe to {len(df)} documents from newspaper: {newspaper}")
    associated_text = grouped[date_key_from_timetable]
    query_embedding = model.encode(sent_sim_template.format(content=associated_text))

    key_dt = pd.to_datetime(date_key_from_timetable)
    buffer = pd.DateOffset(months=month_threshold)

    if go_backwards:
        zone_of_interest = df[
            (df.Date >= key_dt - buffer) & # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]
    else:   
        zone_of_interest = df[
            (df.Date >= key_dt) &          # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]

    print(f"Processing {len(zone_of_interest)} documents in the zone of interest...")
    doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
    doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
    sim_scores = doc_embeddings @ query_embedding.T
    composite, _ = scale_score(sim_scores, zone_of_interest.Date, key_dt, scale_fn, date_weight)

    results = []
    for i in composite.argsort()[::-1]:
        score = composite[i]
        if score >= sim_threshold:
            results.append(zone_of_interest.iloc[i].to_dict())
    return results

def report_result(result, date_key_from_timetable):
    print("-"*40)
    print(f"Date from timetable: {date_key_from_timetable}")
    print(f"Text from timetable:")
    pprint(grouped[date_key_from_timetable])
    print("-"*40)
    print(f"Title: {result['Title']}")
    print(f"Date: {result['Date']}")
    print(f"Newspaper: {result['Newspaper']}")
    pprint(f"{result['chunks']}")
    # print(f"Rating: {result['rating']*5}")
    print("-"*40)

In [28]:
grouped.keys()

dict_keys(['Late 1980s', 'Early 1990s', 'November 1987', '7 March 1988', '10 March 1988', 'April 21, 1989', 'July 1988', 'June 1989', '29 January 1990', 'April 1991', '3 May 1991', 'Sept 1991', '17 February 1992', '27 April 1992', '29 March 1993', 'May 1993', '11 July 1995', '26 March 2001', 'August 2001', 'June 2002', '20 January 2003', '29 August 2003', 'January 2004', '25 March 2004', 'January 2006', 'July 2006', '19 February 2007', 'March 2007', '23 April 2008', 'February 2009', 'March 2009', '20 May 2009', 'April 2010', '14 October 2010', 'January 2011', '8 March 2011', '22 June 2011', 'May 2011', 'October 2011', 'November 2013', 'December 2013', 'April 2014', 'August 2014', 'October 2014', 'March 2015', '25 March 2015', 'October 2015', 'November 2015', 'June 2016', 'January 2017', '11 July 2017', 'August 2017', 'May 2018', '2 July 2018', '5 October 2020', '10 December 2020', '20 May 2024', '31 March 2025', '9 July 2025', '21 July 2025', '30 October 2025', 'November 2025', '31 Dec

In [45]:
date_key = 'Late 1980s'

results = time_bounded_search(
    df,
    model,
    date_key_from_timetable=date_key,
    go_backwards=False,
    month_threshold=12,
    newspaper="The Times"
)

Filtered dataframe to 1146 documents from newspaper: The Times


DateParseError: Unknown datetime string format, unable to parse: Late 1980s

In [34]:
len(results)

2

In [37]:
report_result(results[0], date_key)

----------------------------------------
Date from timetable: April 21, 1989
Text from timetable:
['‘Discovery’/identification of HCV - two seminal papers were published in the '
 'journal Science.',
 'Development of the first antibody test (an ELISA to detect anti-C100-3 '
 'antibodies) by Chiron Corporation, partnering with the CDC']
----------------------------------------
Title: Hepatitis screening is considered;Blood transfusions
Date: 1989-08-25 00:00:00
Newspaper: The Times
('A test that can detect a recently discovered type of hepatitis virus may be '
 'used to screen all blood donations (Our Science Correspondent writes).The '
 'National Blood Transfusion Service is carrying out studies on the '
 'effectiveness of the test, developed in the United States, for hepatitis C, '
 'usually a milder form of the liver disease.The prevalence of the infection '
 'is unknown in Britain, but it is the most common form of hepatitis occurring '
 'after blood transfusions in some developing 

In [3]:
df

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
0,International Companies: Technology - Pounds 2...,1983-02-11,DAVID FISHLOCK,Financial Times,Factiva - HepC,broadsheet,BRITAIN'S biotechnologists are to have a new P...
1,Study confirms fears on spread of Aids,1984-08-31,ANDREW VEITCH,The Guardian,Lexis-Nexis - Times,broadsheet,Fears that the Aids virus is widespread among ...
2,Maternity unit where waste parades the corrido...,1984-10-20,No author in byline,The Guardian,Lexis-Nexis - Times,broadsheet,Dawn in the Rosie maternity unit at the 900-be...
3,Prisons inspector condemns sanitary conditions...,1984-10-24,AILEEN BALLANTYNE,The Guardian,Lexis-Nexis - Times,broadsheet,Gaol conditions fall short of the standard exp...
4,Financial Times Survey: China - Business - Gro...,1984-10-29,No author in byline,Financial Times,Factiva - HepC,broadsheet,CLIMATENorth China (including Beijing) - Winte...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [9]:
doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in df.chunks], show_progress_bar=True, device="mps")

Batches:   0%|          | 0/237 [00:00<?, ?it/s]

In [11]:
import numpy as np
np.save('doc-embeddings811.npy', doc_embeddings)

In [ ]:
doc_embeddings.shape

(7567, 768)

In [ ]:
doc_embeddings[zone_of_interest.index].shape

(121, 768)

In [ ]:
zone_of_interest.shape

(121, 7)

# Full

In [1]:
import json
from tbs import TimeTable, report_result, date_pattern
from pprint import pprint

timetable = TimeTable("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx", date_pattern)
with open("tbs_results811.json", "r") as json_file:
    results_by_date_key = json.load(json_file)

len(timetable.grouped), len(results_by_date_key)

(64, 64)

In [9]:
pprint(list(timetable.grouped.keys()))

['Late 1980s',
 'Early 1990s',
 'November 1987',
 '7 March 1988',
 '10 March 1988',
 'April 21, 1989',
 'July 1988',
 'June 1989',
 '29 January 1990',
 'April 1991',
 '3 May 1991',
 'Sept 1991',
 '17 February 1992',
 '27 April 1992',
 '29 March 1993',
 'May 1993',
 '11 July 1995',
 '26 March 2001',
 'August 2001',
 'June 2002',
 '20 January 2003',
 '29 August 2003',
 'January 2004',
 '25 March 2004',
 'January 2006',
 'July 2006',
 '19 February 2007',
 'March 2007',
 '23 April 2008',
 'February 2009',
 'March 2009',
 '20 May 2009',
 'April 2010',
 '14 October 2010',
 'January 2011',
 '8 March 2011',
 '22 June 2011',
 'May 2011',
 'October 2011',
 'November 2013',
 'December 2013',
 'April 2014',
 'August 2014',
 'October 2014',
 'March 2015',
 '25 March 2015',
 'October 2015',
 'November 2015',
 'June 2016',
 'January 2017',
 '11 July 2017',
 'August 2017',
 'May 2018',
 '2 July 2018',
 '5 October 2020',
 '10 December 2020',
 '20 May 2024',
 '31 March 2025',
 '9 July 2025',
 '21 July 2

In [ ]:
date_key = '20 May 2009' 
result = results_by_date_key[date_key]
report_result(result, date_key, timetable.grouped)

----------------------------------------
Date from timetable: 20 May 2009
['Government response to Lord Archer Report announces no increases of support '
 'for PWHCV.',
 'Funding for a look-back exercise to identify any patients who may have been '
 'infected']
----------------------------------------
Found 30 results
Text from timetable:
**Relevant Article 1**
Title: Compensation call for haemophiliacs
Date: 2009-02-23 00:00:00
Newspaper: Financial Times
('The government should pay compensation and cover future health and insurance '
 'costs for the many hundreds of surviving haemophiliacs left infected by '
 'contaminated National Health Service blood products since the 1970s, an '
 'inquiry concluded .Ministers should also set up a committee to support '
 'haemophiliacs and manage their future treatment to the highest standards, '
 'says a privately funded report written under the chairmanship of Lord Archer '
 'of Sandwell, the former solicitor-general.The conclusions follow length

In [4]:
for date_key in timetable.grouped.keys():
    result = results_by_date_key[date_key]
    if len(result) == 0:
        print(date_key)
        pprint(timetable.grouped[date_key])
        print("-"*40)

10 March 1988
['Macfarlane Trust established']
----------------------------------------
29 January 1990
['Macfarlane I Special Payments Trust established (MFTSP1): £20,000 to each '
 'haemophiliac (or bereaved families)']
----------------------------------------
27 April 1992
['The full details of scheme for non-haemophiliacs/HIV were announced']
----------------------------------------
31 March 2025
['Infected Blood Inquiry second set of regulations came into force to expand '
 'eligibility to "affected" persons (partners, parents, children, etc.) and '
 'introduce supplementary compensation routes']
----------------------------------------
9 July 2025
['The Infected Blood Inquiry released Additional Report on Compensation, '
 'addressing compensation for infected individuals and their families.']
----------------------------------------
21 July 2025
['The Infected Blood Compensation Authority and government official response '
 'confirmed acceptance of recommendations, including furt

In [13]:
for date_key in timetable.grouped.keys():
    result = results_by_date_key[date_key]
    if len(result) > 2 and len(result) > 0:
        print(date_key)
        pprint(timetable.grouped[date_key])
        print("-"*40)

Late 1980s
['Early Treatments: Interferon was the first antiviral treatment for hepatitis '
 'C. significant side effects and lower cure rates.',
 'Victims and their families first started calling for compensation, often on '
 'the grounds of medical negligence, as more evidence is gathered about link '
 'between blood products and HIV/hepatitis infections']
----------------------------------------
Early 1990s
['1987. The Birchgrove Group (often referred to as the BirchGrove Society in '
 'contexts of support), which supports people with haemophilia, HIV, and '
 'Hepatitis C, established in Wales by a small group of patients at the '
 'University Hospital of Wales (UHW) in Cardiff. expanded to a national level.']
----------------------------------------
7 March 1988
['British Liver Trust incorporated and established. (Founded by hepatologist '
 'Professor Dame Sheila Sherlock, leading charity for adults, focusing on '
 'raising awareness, providing support, and campaigning for improved